In [1]:
# ==========================================
# BÀI 9: XỬ LÝ NGOẠI LỆ VÀ GIÁ TRỊ NGOẠI MIỀN (ĐỘC LẬP)
# ==========================================
import pandas as pd
import numpy as np
import re
from google.colab import drive

# 1. Mount Drive và Tải dữ liệu thô
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Nhap_Mon_Hoc_May/Buoi2/Du_lieu_thuc_hanh_chuan_bi_du_lieu.xlsx'

df_raw = pd.read_excel(file_path, sheet_name='Du_lieu_tho')
df_clean = df_raw.copy()

# 2. Tiền xử lý chuẩn hóa cơ bản & chuyển kiểu dữ liệu từ các bài trước
def clean_price(val):
    if pd.isna(val): return np.nan
    v = str(val).lower().strip()
    if 'miễn phí' in v or 'free' in v: return 0.0
    v = re.sub(r'[đ|vnd|\s]', '', v).replace('.', '')
    try: return float(v)
    except: return np.nan

def clean_purchase_count(val):
    if pd.isna(val): return np.nan
    v = str(val).lower().strip().replace('lượt', '').replace(',', '.').strip()
    if 'k' in v:
        try: return float(v.replace('k', '')) * 1000
        except: return np.nan
    try: return float(v)
    except: return np.nan

def clean_discount(val):
    if pd.isna(val): return np.nan
    v = str(val).strip().replace(',', '.')
    if '%' in v:
        try: return float(v.replace('%', '')) / 100.0
        except: return np.nan
    try: return float(v)
    except: return np.nan

df_clean['Price_VND'] = df_clean['Price_VND'].apply(clean_price)
df_clean['Purchase_Count'] = df_clean['Purchase_Count'].apply(clean_purchase_count)
df_clean['Discount'] = df_clean['Discount'].apply(clean_discount)
df_clean['Rating'] = pd.to_numeric(df_clean['Rating'].astype(str).str.replace(',', '.').str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
df_clean['Stock'] = pd.to_numeric(df_clean['Stock'], errors='coerce')

print("=== BẮT ĐẦU THỰC HIỆN BÀI 9 ===\n")


# ------------------------------------------
# Yêu cầu 41: Rating ngoài khoảng [1, 5]
# ------------------------------------------
# Lỗi chắc chắn: Rating < 1 hoặc > 5 -> Chuyển thành NaN và điền median theo Category
df_clean.loc[(df_clean['Rating'] < 1.0) | (df_clean['Rating'] > 5.0), 'Rating'] = np.nan
df_clean['Rating'] = df_clean.groupby('Category')['Rating'].transform(
    lambda x: x.fillna(x.median() if not np.isnan(x.median()) else 4.0)
)
print("-> 41. Đã đưa các giá trị Rating < 1 hoặc > 5 về NaN và điền lại bằng Median theo Category.")


# ------------------------------------------
# Yêu cầu 42: Stock âm và Purchase_Count âm (Lỗi logic không hợp lệ)
# ------------------------------------------
# Không tự động lấy trị tuyệt đối; Chuyển thành NaN và điền median
invalid_stock_mask = df_clean['Stock'] < 0
invalid_purchase_mask = df_clean['Purchase_Count'] < 0

print(f"   - Phát hiện {invalid_stock_mask.sum()} dòng Stock âm.")
print(f"   - Phát hiện {invalid_purchase_mask.sum()} dòng Purchase_Count âm.")

df_clean.loc[invalid_stock_mask, 'Stock'] = np.nan
df_clean.loc[invalid_purchase_mask, 'Purchase_Count'] = np.nan

df_clean['Stock'] = df_clean['Stock'].fillna(df_clean['Stock'].median())
df_clean['Purchase_Count'] = df_clean['Purchase_Count'].fillna(df_clean['Purchase_Count'].median())

print("-> 42. Đã chuyển Stock âm và Purchase_Count âm về NaN (không lấy trị tuyệt đối) và điền Median.")


# ------------------------------------------
# Yêu cầu 43: Xử lý ngoại lệ Stock (Dùng phương pháp IQR)
# ------------------------------------------
Q1 = df_clean['Stock'].quantile(0.25)
Q3 = df_clean['Stock'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

# Tạo cờ đánh dấu outlier cho Stock (ví dụ: Stock = 9999)
df_clean['Stock_Outlier_Flag'] = df_clean['Stock'] > upper_bound

# Cắt ngưỡng (Capping) các giá trị Stock vượt ngưỡng trên về upper_bound thay vì xóa
df_clean['Stock_Capped'] = np.where(df_clean['Stock'] > upper_bound, upper_bound, df_clean['Stock'])

print(f"-> 43. Ngưỡng trên IQR của Stock: {upper_bound}. Đã gắn cờ 'Stock_Outlier_Flag' và thực hiện cắt ngưỡng (Capping).")


# ------------------------------------------
# Yêu cầu 44: Discount ngoài khoảng [0, 1] (Ví dụ: 1.5 hoặc -0.1)
# ------------------------------------------
def fix_discount(val):
    if pd.isna(val): return np.nan
    if val < 0: return np.nan # Discount âm là lỗi -> gán NaN
    if val > 1.0:
        if val <= 100: return val / 100.0 # Ví dụ: 15.0 -> 0.15 hoặc 1.5 -> 0.015/điều chỉnh
        else: return np.nan
    return val

df_clean['Discount'] = df_clean['Discount'].apply(fix_discount)
df_clean['Discount'] = df_clean['Discount'].fillna(0.0) # Điền 0 cho các vị trí lỗi/không có giảm giá

print("-> 44. Đã xử lý Discount âm và ngoài khoảng [0, 1].")


# ------------------------------------------
# Yêu cầu 40 & Báo cáo kết quả
# ------------------------------------------
print("\n[BÁO CÁO PHÂN BIỆT LỖI VÀ GIÁ TRỊ HIẾM (Yêu cầu 40)]:")
print("1. Lỗi chắc chắn (Definite Errors): Rating < 1 hoặc > 5, Stock âm, Purchase_Count âm, Discount âm. Đây là các lỗi vi phạm quy tắc nghiệp vụ, cần chuyển về NaN và làm sạch.")
print("2. Giá trị hiếm/Ngoại lệ có thể hợp lệ (Valid Outliers): Stock = 9999 (kho hàng lớn/đại lý). Xử lý bằng cách tạo cờ 'Stock_Outlier_Flag' và dùng kỹ thuật Capping (Cắt ngưỡng) để giữ quy mô dữ liệu mà không làm lệch mô hình.")

print("\n--- KIỂM TRA LẠI MIỀN GIÁ TRỊ SAU BÀI 9 ---")
print(df_clean[['Rating', 'Stock', 'Purchase_Count', 'Discount']].describe().T[['min', 'max', 'mean']])

Mounted at /content/drive
=== BẮT ĐẦU THỰC HIỆN BÀI 9 ===

-> 41. Đã đưa các giá trị Rating < 1 hoặc > 5 về NaN và điền lại bằng Median theo Category.
   - Phát hiện 1 dòng Stock âm.
   - Phát hiện 1 dòng Purchase_Count âm.
-> 42. Đã chuyển Stock âm và Purchase_Count âm về NaN (không lấy trị tuyệt đối) và điền Median.
-> 43. Ngưỡng trên IQR của Stock: 403.75. Đã gắn cờ 'Stock_Outlier_Flag' và thực hiện cắt ngưỡng (Capping).
-> 44. Đã xử lý Discount âm và ngoài khoảng [0, 1].

[BÁO CÁO PHÂN BIỆT LỖI VÀ GIÁ TRỊ HIẾM (Yêu cầu 40)]:
1. Lỗi chắc chắn (Definite Errors): Rating < 1 hoặc > 5, Stock âm, Purchase_Count âm, Discount âm. Đây là các lỗi vi phạm quy tắc nghiệp vụ, cần chuyển về NaN và làm sạch.
2. Giá trị hiếm/Ngoại lệ có thể hợp lệ (Valid Outliers): Stock = 9999 (kho hàng lớn/đại lý). Xử lý bằng cách tạo cờ 'Stock_Outlier_Flag' và dùng kỹ thuật Capping (Cắt ngưỡng) để giữ quy mô dữ liệu mà không làm lệch mô hình.

--- KIỂM TRA LẠI MIỀN GIÁ TRỊ SAU BÀI 9 ---
                min     